# Graded Activity: Let's Build a Regression Model of Housing Prices
Fill me in

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's setup our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Data
We will develop a model to estimate the housing price based on factors like house area, bedrooms, furnished status, nearness to the main road, etc. The dataset is small; however, its complexity arises because it has strong multicollinearity. 

> __Acknowledgement__
> 
> The housing dataset is adapted from [Kaggle](https://www.kaggle.com/datasets/yasserh/housing-prices-dataset?select=Housing.csv) and was originally used in:
>
> * Harrison, D. and Rubinfeld, D.L. (1978) Hedonic prices and the demand for clean air. J. Environ. Economics and Management 5, 81–102.
> * Belsley D.A., Kuh, E. and Welsch, R.E. (1980) Regression Diagnostics. Identifying Influential Data and Sources of Collinearity. New York: Wiley.

We've encoded [the `MyKaggleHousingPricesDataset()` helper function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyKaggleHousingPricesDataset) which returns the housing dataset [as a `DataFrame` instance](https://dataframes.juliadata.org/stable/).

Let's save the raw (unrangled) data in the `original_dataset::DataFrame` variable:

In [2]:
original_dataset = MyKaggleHousingPricesDataset() # load the *original* dataset

Row,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
,Int64,Int64,Int64,Int64,Int64,String3,String3,String3,String3,String3,Int64,String3,String15
1,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
2,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
3,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
4,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
5,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
6,10850000,7500,3,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
7,10150000,8580,4,3,4,yes,no,no,no,yes,2,yes,semi-furnished
8,10150000,16200,5,3,2,yes,no,no,no,no,0,no,unfurnished
9,9870000,8100,4,1,2,yes,yes,yes,no,yes,2,yes,furnished


Notice that we have several columns (features) that are categorical, i.e., `{furnished | unfurnished | semifurnished}` or `{yes | no`}. We must replace these values with numbers, e.g., `{0 | 1 | 2}`, etc.
 > __Transformation:__ We'll transform the categorical features (including changing types from a `String` to an `Int`) using [the `transform!(...)` method exported by the DataFrames.jl package](https://dataframes.juliadata.org/stable/lib/functions/#DataFrames.transform!). This is an advanced operation, so if this is unclear, please review the [DataFrames.jl documentation](https://dataframes.juliadata.org/stable/lib/functions/#DataFrames.transform!) for more information.

 We'll save the transformed dataset in the `treated_dataset::DataFrame` variable:

In [3]:
treated_dataset = let

    treated_dataset = copy(original_dataset);    
    transform!(treated_dataset, :mainroad => ByRow( x-> (x=="yes" ? 1 : -1)) => :transformed_mainroad);
    transform!(treated_dataset, :guestroom => ByRow( x-> (x=="yes" ? 1 : -1)) => :transformed_guestroom);
    transform!(treated_dataset, :basement => ByRow( x-> (x=="yes" ? 1 : -1)) => :transformed_basement);
    transform!(treated_dataset, :hotwaterheating => ByRow( x-> (x=="yes" ? 1 : -1)) => :transformed_hotwaterheating);
    transform!(treated_dataset, :airconditioning => ByRow( x-> (x=="yes" ? 1 : -1)) => :transformed_airconditioning);
    transform!(treated_dataset, :prefarea => ByRow( x-> (x=="yes" ? 1 : -1)) => :transformed_prefarea);
    transform!(treated_dataset, :furnishingstatus => ByRow( x-> (x=="unfurnished" ? -1 : (x=="semi-furnished" ? 0 : 1))) => :transformed_furnishingstatus);
    transform!(treated_dataset, :price => ByRow(x -> x/1000000.0) => :transformed_price);
    transform!(treated_dataset, :area => ByRow(x -> x/1000.0) => :transformed_area);

    # remove the original columns
    select!(treated_dataset, Not([:mainroad,:guestroom,:basement,:hotwaterheating,:airconditioning,:prefarea,:furnishingstatus,:price, :area]))
end

Row,bedrooms,bathrooms,stories,parking,transformed_mainroad,transformed_guestroom,transformed_basement,transformed_hotwaterheating,transformed_airconditioning,transformed_prefarea,transformed_furnishingstatus,transformed_price,transformed_area
,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64,Float64
1,4,2,3,2,1,-1,-1,-1,1,1,1,13.3,7.42
2,4,4,4,3,1,-1,-1,-1,1,-1,1,12.25,8.96
3,3,2,2,2,1,-1,1,-1,-1,1,0,12.25,9.96
4,4,2,2,3,1,-1,1,-1,1,1,1,12.215,7.5
5,4,1,2,2,1,1,1,-1,1,-1,1,11.41,7.42
6,3,3,1,2,1,-1,1,-1,1,1,0,10.85,7.5
7,4,3,4,2,1,-1,-1,-1,1,1,0,10.15,8.58
8,5,3,2,0,1,-1,-1,-1,-1,-1,-1,10.15,16.2
9,4,1,2,2,1,1,1,-1,1,1,1,9.87,8.1


Now let's split the data set into the system input matrix $\mathbf{X}$ (independent variables, characteristics of the house) and the output vector $\mathbf{y}$ (dependent variables, the house price).

The input matrix $\mathbf{X}$ will contain all the columns except for the `price` column (the output variable). The output vector $\mathbf{y}$ will contain only the `price` column.

In [4]:
X = Matrix(treated_dataset[:, Not(:transformed_price)]); # data matrix: select all the columns *except* price
y = Vector(treated_dataset[:,:transformed_price]); # output vector: select all the price column

Finally, let's partition the data into a `training` and `testing` set so that we can determine how well the model can predict unseen data, i.e., how well the model generalizes.

In [5]:
training, testing = let

    # initialize -
    s = 0.80; # fraction of data for training
    number_of_training_samples = Int(s * size(X,1)); # 80% of the data for training
    i = randperm(size(X,1)); # random permutation of the indices
    training_indices = i[1:number_of_training_samples]; # first 80% of the indices
    testing_indices = i[number_of_training_samples+1:end]; # last 20% of

    # setup data -
    training = (X=X[training_indices, :], y=y[training_indices]);
    testing = (X=X[testing_indices, :], y=y[testing_indices]);

    training, testing;
end;

___

## Task 1: Expected value of the parameters without regularization
We know that the `data matrix` $\mathbf{X}$ is `overdetermined`, i.e., $m>n$ (more equations than unknowns). Thus, we are solving the minimization problem for an unknown parameter estimates $\hat{\theta}$:
$$
\begin{equation*}
\hat{\mathbf{\theta}} = \arg\min_{\mathbf{\theta}} ||~\mathbf{y} - \mathbf{X}\cdot\mathbf{\theta}~||^{2}_{2}
\end{equation*}
$$
where $||\star||^{2}_{2}$ is the square of the p = 2 vector norm. Then, the value of the unknown parameter vector $\mathbf{\theta}$ that minimizes the sum of the squares loss function for an overdetermined system is given by:
$$
\begin{align*}
\hat{\mathbf{\theta}} = \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\mathbf{y}\\
\end{align*}
$$

If we wanted to understand the error in the parameter estimates $\hat{\mathbf{\theta}}$, we could substitute the model $\mathbf{y}$ generated by a linear model with some error term $\mathbf{\epsilon}$:
$$
\begin{align*}
\hat{\mathbf{\theta}} &= \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\underbrace{(\hat{\mathbf{X}}\;\mathbf{\theta} + \mathbf{\epsilon})}_{\text{Model}\;\mathbf{y}}\\
\hat{\mathbf{\theta}} &= \underbrace{\hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\hat{\mathbf{X}}}_{\text{Projection}\;\mathbf{P}}\;\mathbf{\theta} + \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\mathbf{\epsilon}\\
\end{align*}
$$
The __expectation__ removes the error term (we have assumed that the error has zero mean), so that:
$$
\boxed{
\begin{align*}
\mathbb{E}\left[\hat{\mathbf{\theta}}\right] &= \underbrace{\hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\hat{\mathbf{X}}}_{\text{Projection}\;\mathbf{P}}\;\mathbf{\theta}\\
\end{align*}}
$$

In [ ]:
θ̂  = let

    # initialize -

end